In [1]:
# ============================================================
# PREACTIVATION WIDE RESNET — CUTOUT + EMA
#
# Three stages: widths 84, 168, 336; two blocks per stage.
# Trainable parameters: 4,728,330.
#
# Train from scratch using permitted competition training data.
# No pretrained weights, external datasets, or model ensembles.
#
# References:
# Wide ResNet: https://arxiv.org/abs/1605.07146
# Identity mappings: https://arxiv.org/abs/1603.05027
# Cutout: https://arxiv.org/abs/1708.04552
# OneCycle: https://arxiv.org/abs/1708.07120
# ============================================================

import os
import copy
import math
import time
import json
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. CONFIGURATION
# ============================================================

EPOCHS = 200
BATCH = 128
LR = 0.1
WD = 5e-4

VAL_SIZE = 5000
SEED = 42

WIDTHS = (84, 168, 336)
BLOCKS_PER_STAGE = 2

LABEL_SMOOTHING = 0.1
CUTOUT = 16

USE_EMA = True
EMA_DECAY = 0.999
TTA = True
EVAL_BATCH = 128

MODE = "train"                 # "train" or "predict"
DATA_DIRECTORY = ""

# Resume YOUR matching interrupted preactivation experiment.
# Do not use a four-stage model checkpoint here.
RESUME_CHECKPOINT = ""

# Required only when MODE="predict".
PREDICT_CHECKPOINT = ""

assert MODE in {"train", "predict"}
assert EPOCHS >= 2
assert VAL_SIZE == 5000
assert 0 <= CUTOUT <= 32
assert 0 < EMA_DECAY < 1
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings."

dev = torch.device("cuda:0")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False

CONFIG = {
    "experiment": "preact_wrn_cutout_ema_v1",
    "architecture": "three_stage_preactivation_wrn",
    "epochs": EPOCHS,
    "batch_size": BATCH,
    "learning_rate": LR,
    "weight_decay": WD,
    "validation_size": VAL_SIZE,
    "seed": SEED,
    "widths": list(WIDTHS),
    "blocks_per_stage": BLOCKS_PER_STAGE,
    "label_smoothing": LABEL_SMOOTHING,
    "cutout": CUTOUT,
    "color_jitter": 0.0,
    "cutmix": False,
    "use_ema": USE_EMA,
    "ema_decay": EMA_DECAY,
    "ema_buffers": "average_floating_copy_integer",
    "ema_warmup": "min(decay, (1+n)/(10+n))",
    "horizontal_flip_averaging": TTA,
    "optimizer": "SGD",
    "momentum": 0.9,
    "nesterov": True,
    "scheduler": "OneCycleLR",
    "pct_start": 0.15,
    "pretrained": False,
}

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUT = Path("/kaggle/working") / f"preact_wrn_ema_{stamp}"
OUT.mkdir(parents=True, exist_ok=False)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Output:", OUT)
print("One GPU is used.")


# ============================================================
# 2. MODEL — CHECK PARAMETER BUDGET FIRST
# ============================================================

class PreActBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()

        self.bn1 = nn.BatchNorm2d(cin)
        self.c1 = nn.Conv2d(
            cin, cout, 3, stride, 1, bias=False
        )

        self.bn2 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(
            cout, cout, 3, 1, 1, bias=False
        )

        self.sc = None

        if stride != 1 or cin != cout:
            self.sc = nn.Conv2d(
                cin, cout, 1, stride, bias=False
            )

    def forward(self, inputs):
        activated = F.relu(self.bn1(inputs))

        shortcut = (
            inputs if self.sc is None else self.sc(activated)
        )

        residual = self.c1(activated)
        residual = self.c2(F.relu(self.bn2(residual)))

        return shortcut + residual


class PreActWRN(nn.Module):
    def __init__(self, widths, n_blocks, nc=10):
        super().__init__()

        self.stem = nn.Conv2d(
            3, 16, 3, 1, 1, bias=False
        )

        layers = []
        cin = 16

        for stage, width in enumerate(widths):
            for block_index in range(n_blocks):
                stride = (
                    2 if stage > 0 and block_index == 0 else 1
                )

                layers.append(
                    PreActBlock(cin, width, stride)
                )
                cin = width

        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(cin)
        self.fc = nn.Linear(cin, nc)

    def forward(self, inputs):
        features = self.stem(inputs)
        features = self.layers(features)
        features = F.relu(self.bn(features))
        features = F.adaptive_avg_pool2d(features, 1)
        return self.fc(features.flatten(1))


# Fresh initialization: no checkpoint or pretrained weights loaded.
model = (
    PreActWRN(WIDTHS, BLOCKS_PER_STAGE)
    .to(dev)
    .to(memory_format=torch.channels_last)
)

n_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

assert n_params == 4_728_330, (
    f"Unexpected parameter count: {n_params:,}"
)
assert n_params <= 5_000_000

print(f"Trainable parameters: {n_params:,}")
print("Parameter budget check: PASSED")

model.eval()

with torch.inference_mode():
    output = model(
        torch.zeros(2, 3, 32, 32, device=dev)
    )
    assert output.shape == (2, 10)


# ============================================================
# 3. FIND COMPETITION FILES
# ============================================================

if DATA_DIRECTORY:
    ROOT = Path(DATA_DIRECTORY)
else:
    candidates = [
        p.parent
        for p in Path("/kaggle/input").rglob("train_labels.csv")
        if (p.parent / "test.csv").is_file()
        and (p.parent / "sample_submission.csv").is_file()
    ]

    assert len(candidates) == 1, (
        f"Set DATA_DIRECTORY explicitly. Candidates: {candidates}"
    )
    ROOT = candidates[0]

train_df = pd.read_csv(ROOT / "train_labels.csv")
test_df = pd.read_csv(ROOT / "test.csv")
sample = pd.read_csv(ROOT / "sample_submission.csv")

assert {"id", "filename", "label"} <= set(train_df.columns)
assert {"id", "filename"} <= set(test_df.columns)
assert sample.columns.tolist() == ["id", "label"]
assert len(train_df) == 50000

for frame in [train_df, test_df, sample]:
    assert frame.id.is_unique
    assert frame.id.notna().all()

assert train_df[["filename", "label"]].notna().all().all()
assert test_df.filename.notna().all()
assert train_df.filename.is_unique
assert test_df.filename.is_unique

assert train_df.label.between(0, 9).all()
assert (train_df.label == train_df.label.astype(np.int64)).all()

assert len(test_df) == len(sample)
assert set(test_df.id) == set(sample.id)

ordered_test = sample[["id"]].merge(
    test_df[["id", "filename"]],
    on="id",
    how="left",
    sort=False,
    validate="one_to_one",
)

assert ordered_test.filename.notna().all()
assert ordered_test.id.equals(sample.id)

print("Data:", ROOT)


# ============================================================
# 4. LOAD TRAINING IMAGES
# ============================================================

def resolve_folder(name):
    direct = ROOT / name

    if direct.is_dir():
        return direct

    matches = [
        path for path in ROOT.rglob(name)
        if path.is_dir()
    ]

    assert len(matches) == 1, f"Cannot uniquely locate {name}"
    return matches[0]


def load_images(frame, folder):
    base = resolve_folder(folder)
    images = np.empty(
        (len(frame), 32, 32, 3),
        dtype=np.uint8,
    )

    for index, filename in enumerate(frame.filename.astype(str)):
        path = base / filename

        if not path.is_file():
            path = ROOT / filename

        assert path.is_file(), f"Missing image: {path}"

        with Image.open(path) as image:
            pixels = np.array(
                image.convert("RGB"),
                copy=True,
            )

        assert pixels.shape == (32, 32, 3), (
            path, pixels.shape
        )
        images[index] = pixels

    return images


print("Loading permitted training images...")

Xtr = load_images(train_df, "train_images")
ytr = train_df.label.to_numpy(dtype=np.int64)

hasher = hashlib.sha256()
hasher.update(train_df.to_csv(index=False).encode())
hasher.update(memoryview(Xtr))
training_fingerprint = hasher.hexdigest()


# ============================================================
# 5. SAME STRATIFIED VALIDATION SPLIT
# ============================================================

tr_idx, va_idx = train_test_split(
    np.arange(len(ytr)),
    test_size=VAL_SIZE,
    stratify=ytr,
    random_state=SEED,
)

assert not np.intersect1d(tr_idx, va_idx).size

np.savez(
    OUT / "data_split.npz",
    training=tr_idx,
    validation=va_idx,
)

fitting_pixels = Xtr[tr_idx]

mean_np = fitting_pixels.mean(
    axis=(0, 1, 2), dtype=np.float64
) / 255.0

std_np = fitting_pixels.std(
    axis=(0, 1, 2), dtype=np.float64
) / 255.0

del fitting_pixels

assert np.isfinite(mean_np).all()
assert np.isfinite(std_np).all()
assert (std_np > 0).all()


def to_gpu_uint8(images):
    return (
        torch.from_numpy(images)
        .permute(0, 3, 1, 2)
        .contiguous()
        .to(dev)
    )


Xtr_g = to_gpu_uint8(Xtr)
ytr_g = torch.from_numpy(ytr).to(dev)

tr_g = torch.as_tensor(tr_idx, device=dev)
va_g = torch.as_tensor(va_idx, device=dev)

mean = torch.tensor(
    mean_np,
    device=dev,
    dtype=torch.float32,
).view(1, 3, 1, 1)

std = torch.tensor(
    std_np,
    device=dev,
    dtype=torch.float32,
).view(1, 3, 1, 1)

print("Training:", len(tr_idx))
print("Validation:", len(va_idx))
print("Mean:", mean_np)
print("Std:", std_np)


# ============================================================
# 6. GPU AUGMENTATION: FLIP + CROP + CUTOUT
# ============================================================

def normalize(pixels_uint8):
    return (pixels_uint8.float() / 255.0 - mean) / std


def augment(pixels_uint8):
    inputs = normalize(pixels_uint8)
    batch_size = len(inputs)

    flip = torch.rand(batch_size, device=dev) < 0.5

    inputs = torch.where(
        flip[:, None, None, None],
        inputs.flip(-1),
        inputs,
    )

    padded = F.pad(
        inputs,
        (4, 4, 4, 4),
        mode="reflect",
    )

    axis = torch.arange(32, device=dev)

    rows = (
        torch.randint(
            0, 9, (batch_size, 1), device=dev
        ) + axis
    )

    columns = (
        torch.randint(
            0, 9, (batch_size, 1), device=dev
        ) + axis
    )

    inputs = padded[
        torch.arange(
            batch_size, device=dev
        )[:, None, None, None],
        torch.arange(
            3, device=dev
        )[None, :, None, None],
        rows[:, None, :, None],
        columns[:, None, None, :],
    ]

    if CUTOUT > 0:
        cy = torch.randint(
            0, 32, (batch_size, 1, 1), device=dev
        )
        cx = torch.randint(
            0, 32, (batch_size, 1, 1), device=dev
        )

        yy = axis.view(1, 32, 1)
        xx = axis.view(1, 1, 32)
        half = CUTOUT // 2

        # Exactly CUTOUT pixels per side away from borders.
        mask = (
            (yy >= cy - half)
            & (yy < cy - half + CUTOUT)
            & (xx >= cx - half)
            & (xx < cx - half + CUTOUT)
        )

        inputs = inputs.masked_fill(
            mask.unsqueeze(1),
            0,
        )

    return inputs.contiguous(
        memory_format=torch.channels_last
    )


# ============================================================
# 7. EMA
# ============================================================

class EMA:
    def __init__(self, source, decay):
        self.module = copy.deepcopy(source).eval()
        self.module.requires_grad_(False)
        self.decay = decay
        self.updates = 0

    @torch.no_grad()
    def update(self, source):
        self.updates += 1

        decay = min(
            self.decay,
            (1 + self.updates) / (10 + self.updates),
        )

        source_state = source.state_dict()

        # Floating parameters and BatchNorm buffers are averaged.
        # Integer buffers are copied.
        for name, value in self.module.state_dict().items():
            other = source_state[name].detach()

            if value.is_floating_point():
                value.mul_(decay).add_(
                    other,
                    alpha=1.0 - decay,
                )
            else:
                value.copy_(other)


ema = EMA(model, EMA_DECAY) if USE_EMA else None


def serving_model():
    return ema.module if USE_EMA else model


# ============================================================
# 8. INFERENCE AND CHECKPOINT HELPERS
# ============================================================

@torch.inference_mode()
def predict(network, pixels_uint8, indices=None):
    network.eval()
    batches = []

    count = (
        len(pixels_uint8)
        if indices is None
        else len(indices)
    )

    for start in range(0, count, EVAL_BATCH):
        if indices is None:
            pixels = pixels_uint8[start:start + EVAL_BATCH]
        else:
            pixels = pixels_uint8[
                indices[start:start + EVAL_BATCH]
            ]

        inputs = normalize(pixels).contiguous(
            memory_format=torch.channels_last
        )

        with torch.autocast("cuda", dtype=torch.float16):
            probabilities = network(inputs).float().softmax(1)

            if TTA:
                flipped = (
                    network(inputs.flip(-1)).float().softmax(1)
                )
                probabilities = (probabilities + flipped) / 2

        batches.append(probabilities.cpu())

    return torch.cat(batches)


def cpu_state(network):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in network.state_dict().items()
    }


def save_safely(payload, path):
    temporary = path.with_suffix(".temporary")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def selected_checkpoint(weights, epoch, accuracy):
    return {
        "model": weights,
        "config": CONFIG,
        "parameters": n_params,
        "epoch": epoch,
        "validation_accuracy": accuracy,
        "mean": mean_np.tolist(),
        "std": std_np.tolist(),
        "training_fingerprint": training_fingerprint,
        "training_indices": tr_idx,
        "validation_indices": va_idx,
        "weight_source": "ema" if USE_EMA else "raw",
    }


selected_path = OUT / "best.pt"


# ============================================================
# 9. TRAINING WITH RECOVERY
# ============================================================

if MODE == "train":
    decay_parameters = []
    no_decay_parameters = []

    for parameter in model.parameters():
        if parameter.ndim <= 1:
            no_decay_parameters.append(parameter)
        else:
            decay_parameters.append(parameter)

    optimizer = torch.optim.SGD(
        [
            {
                "params": decay_parameters,
                "weight_decay": WD,
            },
            {
                "params": no_decay_parameters,
                "weight_decay": 0.0,
            },
        ],
        lr=LR,
        momentum=0.9,
        nesterov=True,
    )

    steps_per_epoch = math.ceil(len(tr_idx) / BATCH)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LR,
        total_steps=EPOCHS * steps_per_epoch,
        pct_start=0.15,
    )

    scaler = torch.amp.GradScaler("cuda")

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )

    start_epoch = 0
    best_accuracy = -1.0
    best_epoch = 0
    best_weights = None
    history = []

    if RESUME_CHECKPOINT:
        # Only unpickle checkpoints from your own trusted runs.
        saved = torch.load(
            RESUME_CHECKPOINT,
            map_location="cpu",
            weights_only=False,
        )

        assert saved["config"] == CONFIG, (
            "Resume requires identical configuration."
        )
        assert saved["training_fingerprint"] == training_fingerprint
        assert np.array_equal(saved["training_indices"], tr_idx)
        assert np.array_equal(saved["validation_indices"], va_idx)
        assert np.allclose(saved["mean"], mean_np)
        assert np.allclose(saved["std"], std_np)

        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])

        if USE_EMA:
            ema.module.load_state_dict(saved["ema_model"])
            ema.updates = saved["ema_updates"]

        start_epoch = saved["completed_epochs"]
        best_accuracy = saved["best_accuracy"]
        best_epoch = saved["best_epoch"]
        best_weights = saved["best_weights"]
        history = saved["history"]

        random.setstate(saved["python_rng"])
        np.random.set_state(saved["numpy_rng"])
        torch.set_rng_state(saved["torch_rng"])
        torch.cuda.set_rng_state_all(saved["cuda_rng"])

        if best_weights is not None:
            save_safely(
                selected_checkpoint(
                    best_weights,
                    best_epoch,
                    best_accuracy,
                ),
                selected_path,
            )

        print("Resuming at epoch:", start_epoch + 1)

    else:
        print("Training from fresh random initialization.")

    (OUT / "config.json").write_text(
        json.dumps(
            {
                **CONFIG,
                "parameters": n_params,
                "mean": mean_np.tolist(),
                "std": std_np.tolist(),
                "training_fingerprint": training_fingerprint,
                "torch_version": str(torch.__version__),
            },
            indent=2,
        )
    )

    for epoch in range(start_epoch, EPOCHS):
        started = time.time()
        model.train()

        shuffled = tr_g[
            torch.randperm(len(tr_g), device=dev)
        ]

        total_loss = 0.0
        total_examples = 0
        skipped_updates = 0

        for start in range(0, len(shuffled), BATCH):
            batch_indices = shuffled[start:start + BATCH]

            inputs = augment(Xtr_g[batch_indices])
            targets = ytr_g[batch_indices]

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(inputs)
                loss = criterion(logits, targets)

            if not bool(torch.isfinite(loss)):
                raise RuntimeError("Nonfinite training loss.")

            previous_scale = scaler.get_scale()

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if scaler.get_scale() >= previous_scale:
                scheduler.step()

                if USE_EMA:
                    ema.update(model)
            else:
                skipped_updates += 1

            total_loss += loss.item() * len(targets)
            total_examples += len(targets)

        validation_probs = predict(
            serving_model(),
            Xtr_g,
            indices=va_g,
        )

        validation_accuracy = float(
            (
                validation_probs.argmax(1).numpy()
                == ytr[va_idx]
            ).mean()
        )

        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_epoch = epoch + 1
            best_weights = cpu_state(serving_model())

            save_safely(
                selected_checkpoint(
                    best_weights,
                    best_epoch,
                    best_accuracy,
                ),
                selected_path,
            )

        seconds = time.time() - started

        history.append({
            "epoch": epoch + 1,
            "training_loss": total_loss / total_examples,
            "validation_accuracy": validation_accuracy,
            "best_validation_accuracy": best_accuracy,
            "lr_next_step": optimizer.param_groups[0]["lr"],
            "ema_updates": ema.updates if USE_EMA else 0,
            "skipped_amp_updates": skipped_updates,
            "seconds": seconds,
        })

        pd.DataFrame(history).to_csv(
            OUT / "training_history.csv",
            index=False,
        )

        save_safely(
            {
                "model": model.state_dict(),
                "ema_model": (
                    ema.module.state_dict() if USE_EMA else None
                ),
                "ema_updates": ema.updates if USE_EMA else 0,
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "completed_epochs": epoch + 1,
                "best_accuracy": best_accuracy,
                "best_epoch": best_epoch,
                "best_weights": best_weights,
                "history": history,
                "config": CONFIG,
                "mean": mean_np.tolist(),
                "std": std_np.tolist(),
                "training_fingerprint": training_fingerprint,
                "training_indices": tr_idx,
                "validation_indices": va_idx,
                "python_rng": random.getstate(),
                "numpy_rng": np.random.get_state(),
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all(),
            },
            OUT / "last.pt",
        )

        remaining = seconds * (EPOCHS - epoch - 1) / 3600

        print(
            f"Epoch {epoch + 1:03d}/{EPOCHS} | "
            f"Loss {total_loss / total_examples:.4f} | "
            f"Val {validation_accuracy:.2%} | "
            f"Best {best_accuracy:.2%} (ep {best_epoch}) | "
            f"{seconds:.0f}s | Remaining ~{remaining:.2f}h",
            flush=True,
        )

else:
    assert PREDICT_CHECKPOINT, (
        "Set PREDICT_CHECKPOINT to this experiment's best.pt."
    )
    selected_path = Path(PREDICT_CHECKPOINT)


# ============================================================
# 10. LOAD BEST SINGLE MODEL
# ============================================================

checkpoint = torch.load(
    selected_path,
    map_location="cpu",
    weights_only=False,
)

assert checkpoint["config"] == CONFIG
assert checkpoint["training_fingerprint"] == training_fingerprint

model.load_state_dict(checkpoint["model"])
model.eval()

# Final inference uses one network with the selected EMA weights.
if ema is not None:
    del ema
    ema = None
    torch.cuda.empty_cache()

mean = torch.tensor(
    checkpoint["mean"],
    device=dev,
    dtype=torch.float32,
).view(1, 3, 1, 1)

std = torch.tensor(
    checkpoint["std"],
    device=dev,
    dtype=torch.float32,
).view(1, 3, 1, 1)


# ============================================================
# 11. VERIFY VALIDATION AND SAVE PREDICTIONS
# ============================================================

saved_validation = np.asarray(checkpoint["validation_indices"])
validation_ids = train_df.iloc[saved_validation].id.to_numpy()
validation_labels = ytr[saved_validation]

val_probs = predict(
    model,
    Xtr_g,
    indices=torch.as_tensor(saved_validation, device=dev),
).numpy()

val_predictions = val_probs.argmax(axis=1)

verified_accuracy = float(
    (val_predictions == validation_labels).mean()
)

assert abs(
    verified_accuracy - checkpoint["validation_accuracy"]
) < 1e-8

np.save(OUT / "val_probs.npy", val_probs)
np.save(OUT / "val_ids.npy", validation_ids)

pd.DataFrame({
    "id": validation_ids,
    "true_label": validation_labels,
    "prediction": val_predictions,
}).to_csv(
    OUT / "validation_predictions.csv",
    index=False,
)

class_results = pd.DataFrame([
    {
        "class": class_id,
        "count": int((validation_labels == class_id).sum()),
        "accuracy": float(
            (
                val_predictions[validation_labels == class_id]
                == validation_labels[validation_labels == class_id]
            ).mean()
        ),
    }
    for class_id in range(10)
])

class_results.to_csv(
    OUT / "validation_per_class.csv",
    index=False,
)

print("\nSelected epoch:", checkpoint["epoch"])
print(f"Verified validation accuracy: {verified_accuracy:.2%}")
print(class_results.to_string(index=False))


# ============================================================
# 12. EVALUATION INFERENCE
#
# Evaluation images are used for predictions only.
# No test-statistic-based normalization or model selection.
# ============================================================

print("\nLoading evaluation images for inference...")

Xte = load_images(ordered_test, "test_images")
Xte_g = to_gpu_uint8(Xte)

test_probs = predict(model, Xte_g).numpy()

assert np.isfinite(test_probs).all()
assert np.allclose(
    test_probs.sum(axis=1),
    1.0,
    atol=1e-5,
)

predictions = test_probs.argmax(axis=1).astype(np.int64)

np.save(OUT / "test_probs.npy", test_probs)
np.save(OUT / "test_ids.npy", ordered_test.id.to_numpy())


# ============================================================
# 13. WRITE AND CHECK SUBMISSION
# ============================================================

submission = sample[["id"]].copy()
submission["label"] = predictions

assert submission.columns.tolist() == ["id", "label"]
assert len(submission) == len(test_df)
assert submission.id.equals(sample.id)
assert submission.id.is_unique
assert submission.notna().all().all()
assert submission.label.between(0, 9).all()

submission_path = OUT / "submission.csv"
submission.to_csv(submission_path, index=False)

written = pd.read_csv(submission_path)

assert written.columns.tolist() == ["id", "label"]
assert written.id.equals(sample.id)
assert np.array_equal(
    written.label.to_numpy(),
    predictions,
)

(OUT / "results.json").write_text(
    json.dumps(
        {
            "experiment": CONFIG["experiment"],
            "parameters": n_params,
            "selected_epoch": checkpoint["epoch"],
            "validation_accuracy": verified_accuracy,
            "training_images": len(checkpoint["training_indices"]),
            "validation_images": len(saved_validation),
            "weight_source": checkpoint["weight_source"],
            "horizontal_flip_averaging": TTA,
            "submission_rows": len(submission),
            "submission_sha256": hashlib.sha256(
                submission_path.read_bytes()
            ).hexdigest(),
        },
        indent=2,
    )
)

print("\nCOMPLETE")
print("Checkpoint:", selected_path)
print("Submission:", submission_path)
print("Parameters:", f"{n_params:,}")
print("Keep checkpoints, config, split, logs, and probabilities.")
print(submission.head())

from IPython.display import display, FileLink
display(FileLink(str(submission_path)))

GPU: Tesla T4
PyTorch: 2.11.0+cu128
Output: /kaggle/working/preact_wrn_ema_20261101_053059_666002
One GPU is used.
Trainable parameters: 4,728,330
Parameter budget check: PASSED
Data: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl
Loading permitted training images...
Training: 45000
Validation: 5000
Mean: [0.49113735 0.48211119 0.44644899]
Std: [0.24686914 0.24343013 0.26160339]
Training from fresh random initialization.
Epoch 001/200 | Loss 1.7285 | Val 61.18% | Best 61.18% (ep 1) | 28s | Remaining ~1.58h
Epoch 002/200 | Loss 1.4056 | Val 70.50% | Best 70.50% (ep 2) | 26s | Remaining ~1.41h
Epoch 003/200 | Loss 1.2830 | Val 76.70% | Best 76.70% (ep 3) | 26s | Remaining ~1.40h
Epoch 004/200 | Loss 1.1970 | Val 80.50% | Best 80.50% (ep 4) | 26s | Remaining ~1.40h
Epoch 005/200 | Loss 1.1362 | Val 83.06% | Best 83.06% (ep 5) | 26s | Remaining ~1.39h
Epoch 006/200 | Loss 1.0920 | Val 84.70% | Best 84.70% (ep 6) | 26s | Remaining ~1.38h
Epoch 007/200 | Loss 1.0648 | 

/kaggle/working/preact_wrn_ema_20261101_053059_666002/submission.csv